In [1]:
import aioboto3
import io
import pandas as pd
import asyncio
import boto3
from IPython.display import HTML

In [2]:
s3 = boto3.resource('s3')
bucket_name = 'housekg-etl'
prediction_prefix = 'silver/prediction_price_fact/'
price_fact_prefix = 'silver/realty_price_fact/'
realties_prefix =  'silver/realty_dim/'

cat_cols = ["serie", "district", "micro_district", "condition", "toilet"]
num_cols = ["year", "ceiling_height"]
target_col = "sqm_price"
bucket = s3.Bucket(bucket_name)

async def get_df_from_file(bucket_name, file_key, dfs):
    async with aioboto3.Session().client('s3') as s3:
        buffer = io.BytesIO()
        await s3.download_fileobj(bucket_name, file_key, buffer)
        buffer.seek(0)
        dfs.append(pd.read_parquet(buffer))

async def get_df_from_s3(prefix):
    parquet_files = [obj.key for obj in bucket.objects.filter(Prefix=prefix) if obj.key.endswith('.parquet')]
    files_count = len(parquet_files)
    dfs = []
    await asyncio.gather(
        *(get_df_from_file(bucket_name, file_key, dfs) for file_key in parquet_files)
    )
    return pd.concat(dfs) if dfs else None

In [3]:
fact_prices_df = await get_df_from_s3(price_fact_prefix)
predicted_prices_df = await get_df_from_s3(prediction_prefix)
realties_df = await get_df_from_s3(realties_prefix)
print(f"Fact prices df size: {fact_prices_df.shape}")
print(f"Predicted prices df size: {predicted_prices_df.shape}")
print(f"Realties df size: {realties_df.shape}")

Fact prices df size: (1489, 5)
Predicted prices df size: (2952, 5)
Realties df size: (1479, 14)


In [4]:
current_predicted_prices_df = predicted_prices_df[predicted_prices_df['is_current'] == True].rename(columns={'sqm_price': 'predicted_sqm_price'})
current_fact_prices_df = fact_prices_df[fact_prices_df['is_current'] == True].rename(columns={'sqm_price': 'fact_sqm_price'})


In [12]:
anal_df = current_fact_prices_df.merge(current_predicted_prices_df, on='slug').merge(realties_df, on='slug')
anal_df['ratio'] = anal_df['fact_sqm_price'] / anal_df['predicted_sqm_price']
anal_df['url'] = anal_df['slug'].apply(lambda x: f'https://house.kg/details/{x}')


In [13]:
# HTML(anal_df[['url', 'square', 'fact_sqm_price', 'predicted_sqm_price', 'ratio']].sort_values('ratio').to_html(escape=False))
anal_df[['url', 'square', 'fact_sqm_price', 'predicted_sqm_price', 'ratio']].sort_values('ratio')

,url,square,fact_sqm_price,predicted_sqm_price,ratio
666,https://house.kg/details/869153667cf0ef935a448...,189,941.798942,1925.432129,0.489136
1475,https://house.kg/details/1652587679229e258e033...,84,869.047619,1721.763184,0.504743
667,https://house.kg/details/4972417682a07aba3b520...,189,952.380952,1839.180298,0.517829
273,https://house.kg/details/9247140681b37400cf589...,186,908.602151,1740.959961,0.521897
19,https://house.kg/details/848322467bffa983ff834...,414,966.183575,1720.770142,0.561483
...,...,...,...,...,...
1339,https://house.kg/details/926449468286c13ae1a22...,52,1673.076923,1072.317749,1.560244
1330,https://house.kg/details/582205267f8e8f35bbb75...,33,1515.151515,963.092834,1.573214
291,https://house.kg/details/7240297681af497c1e476...,29,1534.482759,910.289429,1.685709
1098,https://house.kg/details/262360167ecaab3102e19...,102,2450.980392,1447.200317,1.693601
